# Metacritic Metadata Cleaning
Cleans the raw movie metadata export: fixes dtypes, fixes mis-encoded text, normalizes title/rating/studio text, adds a primary key, and splits multi-valued columns (`cast`, `genre`, `awards`) into separate junction tables.

In [1]:
import pandas as pd
import numpy as np

## 1. Load data

In [2]:
# Load the raw Excel export
meta = pd.read_excel(r"/Users/huyennguyen/Documents/MDD/SQL/Data Files/metaClean43Brightspace.xlsx")
meta.head()

,url,title,studio,rating,runtime,cast,director,genre,summary,awards,metascore,userscore,RelDate
0,https://www.metacritic.com/movie/!women-art-re...,!Women Art Revolution,Hotwire Productions,| Not Rated,83.0,NaN,Lynn Hershman-Leeson,Documentary,NaN,NaN,70,NaN,2011-06-01
1,https://www.metacritic.com/movie/10-cloverfiel...,10 Cloverfield Lane,Paramount Pictures,| PG-13,104.0,"John Gallagher Jr.,John Goodman,Mary Elizabeth...",Dan Trachtenberg,"Action,Sci-Fi,Drama,Mystery,Thriller,Horror","Waking up from a car accident, a young woman (...","#18MostDiscussedMovieof2016 , #1MostSharedMovi...",76,7.7,2016-03-11
2,https://www.metacritic.com/movie/10-items-or-less,10 Items or Less,Click Star,| R,82.0,"Jonah Hill,Morgan Freeman,Paz Vega",Brad Silberling,"Drama,Comedy,Romance",While researching a role as a supermarket mana...,NaN,54,5.8,2006-12-01
3,https://www.metacritic.com/movie/10-years,10 Years,Anchor Bay Entertainment,| R,100.0,"Channing Tatum,Chris Pratt,Jenna Dewan",Jamie Linden,"Drama,Comedy,Romance",NaN,NaN,61,6.9,2012-09-14
4,https://www.metacritic.com/movie/100-bloody-acres,100 Bloody Acres,Music Box Films,| Not Rated,91.0,NaN,Cameron Cairnes,"Horror,Comedy",Reg and Lindsay run an organic fertilizer busi...,NaN,63,7.5,2013-06-28


In [3]:
# Quick profile of the raw data
print(meta.info())
print(meta.describe())
print("Shape:", meta.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11364 entries, 0 to 11363
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   url        11364 non-null  object        
 1   title      11364 non-null  object        
 2   studio     11014 non-null  object        
 3   rating     10297 non-null  object        
 4   runtime    11109 non-null  float64       
 5   cast       7662 non-null   object        
 6   director   11350 non-null  object        
 7   genre      11344 non-null  object        
 8   summary    5467 non-null   object        
 9   awards     4387 non-null   object        
 10  metascore  11364 non-null  int64         
 11  userscore  9259 non-null   float64       
 12  RelDate    11364 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(1), object(9)
memory usage: 1.1+ MB
None
            runtime     metascore    userscore                        RelDate
count  11109.000000  1

In [4]:
# Check for fully duplicated rows
print("Duplicate rows:", meta.duplicated().sum())

Duplicate rows: 0


In [5]:
# Missing values per column: count + percentage
for col in meta.columns:
    missing_pct = meta[col].isnull().mean() * 100
    print(f"{col}: {meta[col].isnull().sum()} missing ({missing_pct:.2f}%)")

url: 0 missing (0.00%)
title: 0 missing (0.00%)
studio: 350 missing (3.08%)
rating: 1067 missing (9.39%)
runtime: 255 missing (2.24%)
cast: 3702 missing (32.58%)
director: 14 missing (0.12%)
genre: 20 missing (0.18%)
summary: 5897 missing (51.89%)
awards: 6977 missing (61.40%)
metascore: 0 missing (0.00%)
userscore: 2105 missing (18.52%)
RelDate: 0 missing (0.00%)


## 2. Data type conversion
Convert free-text object columns to pandas' `string` dtype for consistent `.str` handling.

In [6]:
for col in ["title", "studio", "rating", "director"]:
    meta[col] = meta[col].astype("string")

meta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11364 entries, 0 to 11363
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   url        11364 non-null  object        
 1   title      11364 non-null  string        
 2   studio     11014 non-null  string        
 3   rating     10297 non-null  string        
 4   runtime    11109 non-null  float64       
 5   cast       7662 non-null   object        
 6   director   11350 non-null  string        
 7   genre      11344 non-null  object        
 8   summary    5467 non-null   object        
 9   awards     4387 non-null   object        
 10  metascore  11364 non-null  int64         
 11  userscore  9259 non-null   float64       
 12  RelDate    11364 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(1), object(5), string(4)
memory usage: 1.1+ MB


## 3. Fix mis-encoded text (mojibake)
Several text columns were double-encoded (UTF-8 bytes misread as Latin-1), e.g. `GÃ©rard` instead of `Gérard`. Only columns that actually contain the artifact (checked against the raw data) are fixed.

In [7]:
def try_encoding(text):
    # Reverse a UTF-8 -> Latin-1 mis-decode; leave normal text untouched.
    if pd.isna(text):
        return text
    try:
        return text.encode("latin1").decode("utf-8")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return text

for col in ["title", "studio", "cast", "director", "summary"]:
    meta[col] = meta[col].apply(try_encoding)

# Spot-check: rows that still contain the mojibake marker should now be fixed
print("Remaining mojibake in cast:", meta["cast"].str.contains("Ã", na=False).sum())

Remaining mojibake in cast: 1


## 4. Clean text formatting

In [8]:
# Strip leading/trailing whitespace from every free-text column
text_cols = ["title", "studio", "rating", "cast", "director", "genre", "summary", "awards"]
for col in text_cols:
    meta[col] = meta[col].str.strip()

In [9]:
# Normalize titles: lowercase, spell out "&" as "and", drop remaining punctuation
meta["title"] = meta["title"].str.lower()
meta["title"] = meta["title"].str.replace("&", "and", regex=False)
meta["title"] = meta["title"].str.replace(r"[^\w\s]", "", regex=True)
meta["title"] = meta["title"].str.strip()

In [10]:
# rating values look like "| PG-13" (and a few typos like "PG--13", "PG-13`").
# Strip all punctuation so only the alphanumeric rating code remains, then re-strip whitespace
# left behind by removing the leading "| ".
meta["rating"] = meta["rating"].str.replace(r"[^\w\s]", "", regex=True).str.strip()

print(meta["rating"].value_counts(dropna=False))

rating
R            3515
Not Rated    2960
PG13         2031
<NA>         1067
PG            816
Unrated       384
TVMA          200
NR            128
G             124
TV14           56
NC17           40
TVPG           23
TVG             7
Open            5
Approved        4
M               2
MA17            1
MPG             1
Name: count, dtype: Int64


In [11]:
# studio names like "Weinstein Company, The" should read "The Weinstein Company"
meta["studio"] = meta["studio"].str.replace(
    r"^(.*),\s*(The|A|An)$", r"\2 \1", regex=True
)

## 5. Add a primary key
`title` alone has 219 duplicate values across different movies, so it can't safely be used as a join key for the junction tables below — a stable `movie_id` is needed instead.

In [12]:
meta = meta.reset_index(drop=True)
meta.insert(0, "movie_id", range(1, len(meta) + 1))

## 6. Flag runtime outliers
Runtime tops out at 808 minutes — implausible for a single feature, so flag rows above 300 minutes for manual review rather than guessing a correction.

In [13]:
meta["runtime_outlier"] = meta["runtime"] > 300
print("Runtime outliers flagged:", meta["runtime_outlier"].sum())

Runtime outliers flagged: 3


## 7. Split multi-valued columns into junction tables
`cast`, `genre`, and `awards` hold comma-separated lists in a single cell. Split each into a Python list, then explode into its own long-format table keyed by `movie_id` — **not** chained together, since exploding three list columns on the same row would cross-multiply them into a combinatorial explosion (e.g. 3 cast x 6 genre x N awards).

In [14]:
# Split into lists; awards has inconsistent spacing around commas so uses a regex split
meta["cast_list"] = meta["cast"].str.split(",")
meta["genre_list"] = meta["genre"].str.split(",")
meta["awards_list"] = meta["awards"].str.split(r"\s*,\s*", regex=True)

# Strip whitespace from each item in the lists
for col in ["cast_list", "genre_list", "awards_list"]:
    meta[col] = meta[col].apply(lambda lst: [x.strip() for x in lst] if isinstance(lst, list) else lst)

In [15]:
# Junction table: one row per (movie, genre)
movie_genre = (
    meta[["movie_id", "title", "genre_list"]]
    .explode("genre_list")
    .rename(columns={"genre_list": "genre"})
    .dropna(subset=["genre"])
    .reset_index(drop=True)
)
movie_genre.head(10)

,movie_id,title,genre
0,1,women art revolution,Documentary
1,2,10 cloverfield lane,Action
2,2,10 cloverfield lane,Sci-Fi
3,2,10 cloverfield lane,Drama
4,2,10 cloverfield lane,Mystery
5,2,10 cloverfield lane,Thriller
6,2,10 cloverfield lane,Horror
7,3,10 items or less,Drama
8,3,10 items or less,Comedy
9,3,10 items or less,Romance


In [16]:
# Junction table: one row per (movie, cast member)
movie_cast = (
    meta[["movie_id", "title", "cast_list"]]
    .explode("cast_list")
    .rename(columns={"cast_list": "cast"})
    .dropna(subset=["cast"])
    .reset_index(drop=True)
)
movie_cast.head(10)

,movie_id,title,cast
0,2,10 cloverfield lane,John Gallagher Jr.
1,2,10 cloverfield lane,John Goodman
2,2,10 cloverfield lane,Mary Elizabeth Winstead
3,3,10 items or less,Jonah Hill
4,3,10 items or less,Morgan Freeman
5,3,10 items or less,Paz Vega
6,4,10 years,Channing Tatum
7,4,10 years,Chris Pratt
8,4,10 years,Jenna Dewan
9,8,10000 bc,Camilla Belle


In [17]:
# Junction table: one row per (movie, award)
movie_awards = (
    meta[["movie_id", "title", "awards_list"]]
    .explode("awards_list")
    .rename(columns={"awards_list": "awards"})
    .dropna(subset=["awards"])
    .reset_index(drop=True)
)
movie_awards.head(10)

,movie_id,title,awards
0,2,10 cloverfield lane,#18MostDiscussedMovieof2016
1,2,10 cloverfield lane,#1MostSharedMovieof2016
2,8,10000 bc,#23MostDiscussedMovieof2008
3,8,10000 bc,#27MostSharedMovieof2008
4,12,102 dalmatians,#73MostDiscussedMovieof2000
5,17,12,#97BestMovieof2009
6,17,12,#18MostSharedMovieof2009
7,22,12 rounds,#71MostSharedMovieof2009
8,23,12 strong,#77MostDiscussedMovieof2018
9,23,12 strong,#21MostSharedMovieof2018


## 8. Build the cleaned movies table
One row per movie — the multi-valued columns now live in the junction tables above, so they're dropped here to avoid duplicating movie-level data.

In [18]:
meta_cleaned = meta.drop(columns=["cast", "genre", "awards", "cast_list", "genre_list", "awards_list"])
meta_cleaned.head(10)

,movie_id,url,title,studio,rating,runtime,director,summary,metascore,userscore,RelDate,runtime_outlier
0,1,https://www.metacritic.com/movie/!women-art-re...,women art revolution,Hotwire Productions,Not Rated,83.0,Lynn Hershman-Leeson,NaN,70,NaN,2011-06-01,False
1,2,https://www.metacritic.com/movie/10-cloverfiel...,10 cloverfield lane,Paramount Pictures,PG13,104.0,Dan Trachtenberg,"Waking up from a car accident, a young woman (...",76,7.7,2016-03-11,False
2,3,https://www.metacritic.com/movie/10-items-or-less,10 items or less,Click Star,R,82.0,Brad Silberling,While researching a role as a supermarket mana...,54,5.8,2006-12-01,False
3,4,https://www.metacritic.com/movie/10-years,10 years,Anchor Bay Entertainment,R,100.0,Jamie Linden,NaN,61,6.9,2012-09-14,False
4,5,https://www.metacritic.com/movie/100-bloody-acres,100 bloody acres,Music Box Films,Not Rated,91.0,Cameron Cairnes,Reg and Lindsay run an organic fertilizer busi...,63,7.5,2013-06-28,False
5,6,https://www.metacritic.com/movie/100-streets,100 streets,Samuel Goldwyn Films,<NA>,93.0,Jim O'Hanlon,NaN,44,6.1,2017-01-13,False
6,7,https://www.metacritic.com/movie/1000-times-go...,1000 times good night,Film Movement,Not Rated,117.0,Erik Poppe,NaN,57,6.8,2014-10-24,False
7,8,https://www.metacritic.com/movie/10000-bc,10000 bc,Warner Bros. Pictures,PG13,109.0,Roland Emmerich,NaN,34,4.6,2008-03-07,False
8,9,https://www.metacritic.com/movie/10000-km,10000 km,Broad Green Pictures,R,99.0,Carlos Marques-Marcet,"Two people in love, two apartments - one in Ba...",75,7.4,2015-07-10,False
9,10,https://www.metacritic.com/movie/1001-grams,1001 grams,Kino Lorber,Not Rated,93.0,Bent Hamer,When Norwegian scientist Marie attends a semin...,65,NaN,2015-05-08,False


## 9. Final quality check and export

In [19]:
print("FINAL DATA QUALITY CHECK")
print("Movies:", meta_cleaned.shape)
print("Unique movie_id:", meta_cleaned["movie_id"].is_unique)
print("movie_genre rows:", movie_genre.shape)
print("movie_cast rows:", movie_cast.shape)
print("movie_awards rows:", movie_awards.shape)
print("Runtime outliers flagged:", meta_cleaned["runtime_outlier"].sum())

FINAL DATA QUALITY CHECK
Movies: (11364, 12)
Unique movie_id: True
movie_genre rows: (27055, 3)
movie_cast rows: (48314, 3)
movie_awards rows: (6409, 3)
Runtime outliers flagged: 3


In [20]:
# Export the cleaned movie table and its junction tables as separate sheets
with pd.ExcelWriter("meta_cleaned_v1.xlsx") as writer:
    meta_cleaned.to_excel(writer, sheet_name="movies", index=False)
    movie_genre.to_excel(writer, sheet_name="movie_genre", index=False)
    movie_cast.to_excel(writer, sheet_name="movie_cast", index=False)
    movie_awards.to_excel(writer, sheet_name="movie_awards", index=False)

print("Saved successfully")
print(meta_cleaned.shape)

Saved successfully
(11364, 12)
